# SAM2 vs UltraSAM2 — Side-by-Side Video

Generates one MP4 per patient showing all annotated frames as a 2×2 mosaic:

| | Left | Right |
|---|---|---|
| **Top** | SAM2 (raw input) | UltraSAM2 (raw input) |
| **Bottom** | SAM2 (denoised input) | UltraSAM2 (denoised input) |

Each quadrant shows the predicted mask overlaid in green on the source image.  
IoU vs ground truth is shown in each panel title.

**Output:** `patient_1_sam2_vs_ultrasam2.mp4`, `patient_2_…`, `patient_3_…`

In [81]:
import os
import re
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw): return it

In [82]:

# ── Paths ─────────────────────────────────────────────────────────────────────
GT_ROOT       = r"D:\annotated_pn"
SAM2_RAW_ROOT = r"D:\sam2-results"
SAM2_DN_ROOT  = r"D:\sam2-results\denoised"
US2_RAW_ROOT  = r"D:\ultrasam2-results\inference1"
US2_DN_ROOT   = r"D:\ultrasam2-results\inference1_dn"
RAW_SRC_ROOT  = r"D:\dissertation-results\cropped_pngs3"
DN_SRC_ROOT   = r"D:\dissertation-results\speckle2self-results\git_inf\260720-cropped4\epoch199"

OUT_DIR = os.getcwd()

PATIENTS = {
    "Almeida": "AlmeidaDenoised2_nerve",
    "Borges":  "BorgesDenoised2_nerve",
    "Brito":   "BritoDenoised2_nerve",
}
LABEL_MAP = {"Almeida": "Patient 1", "Borges": "Patient 2", "Brito": "Patient 3"}

FPS     = 12     # video frames per second
FIGSIZE = (12, 9)  # inches — kept constant so all frames are the same size
DPI     = 100
STEP    = 1        # set to 2 to use every other frame and halve file size


In [83]:

# ── Helpers ───────────────────────────────────────────────────────────────────
def frame_key_from_gt(fname: str) -> str:
    s = re.sub(r'^Label_\d+_', '', fname)
    return s.replace('_denoised.png', '').replace('.png', '')


def index_masks(mask_dir: str, suffixes: list) -> dict:
    """Build {frame_key: path} from a masks directory."""
    if not os.path.exists(mask_dir):
        return {}
    idx = {}
    for f in os.listdir(mask_dir):
        if not f.endswith('.png'):
            continue
        key = f
        for suf in suffixes:
            if key.endswith(suf):
                key = key[:-len(suf)]
                break
        idx[key] = os.path.join(mask_dir, f)
    return idx


def load_binary(path: str) -> np.ndarray:
    return (np.array(Image.open(path).convert('L')) > 0)


def make_overlay(img_arr, mask_arr, color=(0, 200, 60), alpha=0.5):
    out = img_arr.copy().astype(np.float32)
    out[mask_arr.astype(bool)] = (
        out[mask_arr.astype(bool)] * (1 - alpha)
        + np.array(color, dtype=np.float32) * alpha
    )
    return out.astype(np.uint8)


def sort_key(frame_key: str) -> tuple:
    """Sort by all numbers in the filename: first number = video, second = frame."""
    nums = tuple(int(x) for x in re.findall(r'\d+', frame_key))
    return nums if nums else (0,)


In [84]:

# ── Build unified frame table (ALL frames, GT matched where available) ─────────
records = []

for patient, gt_folder in PATIENTS.items():
    # Drive from the raw source directory — every frame in the full scan
    src_dir  = os.path.join(RAW_SRC_ROOT, patient, "cross-section")
    all_keys = sorted(
        [f[:-4] for f in os.listdir(src_dir) if f.endswith('.png')],
        key=sort_key
    )

    gt_dir   = os.path.join(GT_ROOT, gt_folder)
    gt_index = {
        frame_key_from_gt(f): os.path.join(gt_dir, f)
        for f in os.listdir(gt_dir) if f.endswith('.png')
    }

    s2_raw  = index_masks(os.path.join(SAM2_RAW_ROOT, patient, "masks"),
                           ['_mask.png'])
    s2_dn   = index_masks(os.path.join(SAM2_DN_ROOT, patient, "masks"),
                           ['_denoised_mask.png', '_mask.png'])
    us2_raw = index_masks(os.path.join(US2_RAW_ROOT, patient, "masks"),
                           ['_mask.png'])
    us2_dn  = index_masks(os.path.join(US2_DN_ROOT, patient.lower(), "masks"),
                           ['_denoised_mask.png', '_mask.png'])

    for fk in all_keys:
        records.append({
            "patient":      patient,
            "frame_key":    fk,
            "gt_path":      gt_index.get(fk),       # None if not annotated
            "s2_raw_mask":  s2_raw.get(fk),
            "s2_dn_mask":   s2_dn.get(fk),
            "us2_raw_mask": us2_raw.get(fk),
            "us2_dn_mask":  us2_dn.get(fk),
            "raw_src": os.path.join(RAW_SRC_ROOT, patient, "cross-section", f"{fk}.png"),
            "dn_src":  os.path.join(DN_SRC_ROOT, patient.lower(), f"{fk}_denoised.png"),
        })

df = pd.DataFrame(records)
print(f"Total frames: {len(df)}")
print(df.groupby('patient').size().rename('frames'))
print()
for col in ['s2_raw_mask', 's2_dn_mask', 'us2_raw_mask', 'us2_dn_mask']:
    n_miss = df[col].isna().sum()
    status = f"[WARN] {n_miss} missing ({100*n_miss/len(df):.0f}%)" if n_miss else "OK — all frames covered"
    print(f"  {col:15s}: {status}")
print(f"\n  Annotated frames: {df['gt_path'].notna().sum()} / {len(df)}")


Total frames: 767
patient
Almeida    249
Borges     269
Brito      249
Name: frames, dtype: int64

  s2_raw_mask    : OK — all frames covered
  s2_dn_mask     : OK — all frames covered
  us2_raw_mask   : OK — all frames covered
  us2_dn_mask    : OK — all frames covered

  Annotated frames: 134 / 767


In [85]:

# IoU computation skipped — titles show model names only


In [86]:

# ── Frame renderer ────────────────────────────────────────────────────────────
def render_frame(row) -> np.ndarray:
    """Returns a fixed-size H×W×3 uint8 array for one frame."""
    raw_src = np.array(Image.open(row['raw_src']).convert('RGB'))

    dn_path = row['dn_src']
    dn_src  = np.array(Image.open(dn_path).convert('RGB')) \
              if os.path.exists(dn_path) else raw_src

    blank_raw = np.zeros(raw_src.shape[:2], bool)
    blank_dn  = np.zeros(dn_src.shape[:2],  bool)

    def _load(path, blank):
        return load_binary(path) if pd.notna(path) and path else blank

    s2_raw_mask  = _load(row['s2_raw_mask'],  blank_raw)
    us2_raw_mask = _load(row['us2_raw_mask'], blank_raw)
    s2_dn_mask   = _load(row['s2_dn_mask'],   blank_dn)
    us2_dn_mask  = _load(row['us2_dn_mask'],  blank_dn)

    fig, axes = plt.subplots(2, 2, figsize=FIGSIZE, dpi=DPI)
    fig.patch.set_facecolor('#1a1a1a')
    fig.suptitle(LABEL_MAP[row['patient']], fontsize=14, fontweight='bold',
                 color='white', y=0.995)

    panels = [
        (axes[0, 0], make_overlay(raw_src, s2_raw_mask),  "SAM2"),
        (axes[0, 1], make_overlay(raw_src, us2_raw_mask), "UltraSAM2"),
        (axes[1, 0], make_overlay(dn_src,  s2_dn_mask),   "SAM2 (denoised)"),
        (axes[1, 1], make_overlay(dn_src,  us2_dn_mask),  "UltraSAM2 (denoised)"),
    ]

    for ax, img, title in panels:
        ax.imshow(img)
        ax.set_title(title, fontsize=11, color='white', pad=4)
        ax.axis('off')
        ax.set_facecolor('#1a1a1a')

    fig.text(0.01, 0.73, 'Raw',      va='center', ha='left',
             fontsize=10, color='#aaaaaa', rotation=90)
    fig.text(0.01, 0.27, 'Denoised', va='center', ha='left',
             fontsize=10, color='#aaaaaa', rotation=90)

    plt.tight_layout(rect=[0.02, 0, 1, 0.97])

    fig.canvas.draw()
    buf = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    plt.close(fig)
    return buf


In [87]:
# ── Video writer ──────────────────────────────────────────────────────────────
def save_video(frames: list, out_path: str, fps: int = FPS) -> None:
    """Save a list of H×W×3 uint8 frames as MP4 (tries imageio then cv2)."""
    e_iio = None
    try:
        import imageio
        with imageio.get_writer(out_path, fps=fps, quality=8, macro_block_size=None) as writer:
            for f in frames:
                writer.append_data(f)
        print(f"  [imageio] Saved → {out_path}")
        return
    except Exception as exc:
        e_iio = exc
        print(f"  imageio failed ({exc}), trying cv2...")

    try:
        import cv2
        h, w = frames[0].shape[:2]
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        vw = cv2.VideoWriter(out_path, fourcc, float(fps), (w, h))
        for f in frames:
            vw.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
        vw.release()
        print(f"  [cv2] Saved → {out_path}")
    except Exception as e_cv2:
        raise RuntimeError(
            f"Both video backends failed.\n  imageio: {e_iio}\n  cv2: {e_cv2}"
        )

In [88]:

# ── Generate one video per patient ────────────────────────────────────────────
for patient in PATIENTS:
    patient_df = df[df['patient'] == patient].iloc[::STEP].reset_index(drop=True)
    label      = LABEL_MAP[patient]
    slug       = label.replace(' ', '_').lower()
    out_path   = os.path.join(OUT_DIR, f"{slug}_sam2_vs_ultrasam2.mp4")

    n_total = (df['patient'] == patient).sum()
    print(f"\n{label}: {len(patient_df)} frames (step={STEP}, {n_total} total)...")
    frames = [
        render_frame(row)
        for _, row in tqdm(patient_df.iterrows(), total=len(patient_df), desc=label)
    ]

    save_video(frames, out_path)
    print(f"  {len(frames)} frames @ {FPS} fps → {len(frames)/FPS:.1f}s")



Patient 1: 249 frames (step=1, 249 total)...


Patient 1: 100%|██████████| 249/249 [00:27<00:00,  8.99it/s]


  imageio failed (TiffWriter.write() got an unexpected keyword argument 'fps'), trying cv2...
  [cv2] Saved → d:\graphs-n-tables\patient_1_sam2_vs_ultrasam2.mp4
  249 frames @ 12 fps → 20.8s

Patient 2: 269 frames (step=1, 269 total)...


Patient 2: 100%|██████████| 269/269 [00:34<00:00,  7.86it/s]


  imageio failed (TiffWriter.write() got an unexpected keyword argument 'fps'), trying cv2...
  [cv2] Saved → d:\graphs-n-tables\patient_2_sam2_vs_ultrasam2.mp4
  269 frames @ 12 fps → 22.4s

Patient 3: 249 frames (step=1, 249 total)...


Patient 3: 100%|██████████| 249/249 [00:34<00:00,  7.22it/s]


  imageio failed (TiffWriter.write() got an unexpected keyword argument 'fps'), trying cv2...
  [cv2] Saved → d:\graphs-n-tables\patient_3_sam2_vs_ultrasam2.mp4
  249 frames @ 12 fps → 20.8s
